In [0]:
import json

VOLUME = "/Volumes/workspace/default/ecommerce_data"

# Charger les recommandations ALS
recs = spark.read.parquet(f"{VOLUME}/recommendations_als")

# Charger les mappings
with open(f"{VOLUME}/user_mapping.json") as f:
    user_mapping = json.load(f)

with open(f"{VOLUME}/item_mapping.json") as f:
    item_mapping = json.load(f)

# Convertir en dictionnaire simple
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, collect_list

# Top-10 par user
window_spec = Window.partitionBy("user_idx").orderBy(col("score").desc())

top10 = recs \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("top10_items"))

# Construire le dictionnaire final
recs_dict = {}
for row in top10.collect():
    user_id   = user_mapping[row["user_idx"]]
    products  = [item_mapping[idx] for idx in row["top10_items"]]
    recs_dict[user_id] = products

print(f" {len(recs_dict):,} users dans les recommandations")

# Sauvegarder en JSON
with open(f"{VOLUME}/recommendations_api.json", "w") as f:
    json.dump(recs_dict, f)

print(" Fichier JSON sauvegardé !")


 100 users dans les recommandations
 Fichier JSON sauvegardé !
